# 🔧 Engenharia de Dados
## Crime Data from 2020 to Present - Los Angeles

Este notebook realiza as operações de engenharia de dados no dataset de crimes de Los Angeles.

### Etapas:
1. Carregamento dos dados
2. Exploração inicial
3. Limpeza de dados
4. Transformações
5. Feature engineering
6. Exportação dos dados processados

In [ ]:
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')

# Importar módulos locais
import sys
sys.path.append('..')
from src.data_processing import *

print('✅ Bibliotecas carregadas com sucesso!')

## 1. Carregamento dos Dados

In [ ]:
# Carregar dataset
# Para testes iniciais, use sample_size para carregar uma amostra
DATA_PATH = '../data/raw/Crime_Data_from_2020_to_Present.csv'

# Carregar amostra para desenvolvimento (descomente para dados completos)
df = load_crime_data(DATA_PATH, sample_size=100000)
# df = load_crime_data(DATA_PATH)  # Dados completos

print(f"\nShape: {df.shape}")
print(f"Memória utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Visualizar primeiras linhas
df.head()

In [ ]:
# Informações sobre o dataset
df.info()

## 2. Exploração Inicial

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Verificar valores nulos
null_counts = df.isnull().sum()
null_percentage = (null_counts / len(df) * 100).round(2)

null_df = pd.DataFrame({
    'Valores Nulos': null_counts,
    'Porcentagem (%)': null_percentage
}).sort_values('Porcentagem (%)', ascending=False)

null_df[null_df['Valores Nulos'] > 0]

In [ ]:
# Visualizar valores nulos
plt.figure(figsize=(14, 6))
null_cols = null_df[null_df['Valores Nulos'] > 0]
plt.barh(null_cols.index, null_cols['Porcentagem (%)'])
plt.xlabel('Porcentagem de Valores Nulos (%)')
plt.title('Colunas com Valores Nulos')
plt.tight_layout()
plt.show()

In [ ]:
# Tipos de dados
df.dtypes

## 3. Limpeza dos Dados

In [ ]:
# Converter colunas de data
df = clean_dates(df)

# Verificar conversão
print("Tipos após conversão:")
print(df[['Date Rptd', 'DATE OCC', 'HOUR']].dtypes)

In [ ]:
# Tratar valores ausentes
print(f"Linhas antes: {len(df):,}")
df = handle_missing_values(df, strategy='drop')
print(f"Linhas depois: {len(df):,}")

In [ ]:
# Normalizar coordenadas geográficas
print(f"Linhas antes: {len(df):,}")
df = normalize_coordinates(df)
print(f"Linhas depois: {len(df):,}")

In [ ]:
# Verificar idade das vítimas
print("Estatísticas de idade:")
print(df['Vict Age'].describe())

# Remover idades inválidas
df = df[(df['Vict Age'] >= 0) & (df['Vict Age'] <= 120)]
print(f"\nLinhas após filtro de idade: {len(df):,}")

## 4. Feature Engineering

In [ ]:
# Criar features derivadas
df = create_derived_features(df)

# Verificar novas colunas
new_cols = ['YEAR', 'MONTH', 'DAY_OF_WEEK', 'DAY_NAME', 'IS_WEEKEND', 'PERIOD', 'AGE_GROUP', 'IS_VIOLENT']
df[new_cols].head()

In [ ]:
# Codificar variáveis categóricas
categorical_cols = ['AREA NAME', 'Vict Sex', 'Vict Descent']
df, encodings = encode_categorical(df, categorical_cols)

print("Colunas codificadas criadas:")
for col in categorical_cols:
    print(f"  - {col}_encoded")

In [ ]:
# Resumo do dataset processado
summary = get_data_summary(df)
print(f"Total de registros: {summary['total_records']:,}")
print(f"Total de colunas: {summary['total_columns']}")

In [ ]:
# Verificar dataset final
df.info()

## 5. Exportação dos Dados Processados

In [ ]:
# Criar diretório se não existir
import os
os.makedirs('../data/processed', exist_ok=True)

# Salvar dataset processado
output_path = '../data/processed/crime_data_processed.csv'
df.to_csv(output_path, index=False)
print(f"✅ Dataset processado salvo em: {output_path}")
print(f"   Tamanho: {os.path.getsize(output_path) / 1024**2:.2f} MB")

In [ ]:
# Salvar encodings para uso posterior
import json

# Converter para JSON serializável
encodings_json = {k: {str(k2): v2 for k2, v2 in v.items()} for k, v in encodings.items()}

with open('../data/processed/encodings.json', 'w') as f:
    json.dump(encodings_json, f, indent=2)
    
print("✅ Encodings salvos em: ../data/processed/encodings.json")

## 📊 Resumo da Engenharia de Dados

### Operações realizadas:
- ✅ Carregamento dos dados brutos
- ✅ Conversão de tipos de dados (datas, numéricos)
- ✅ Tratamento de valores nulos
- ✅ Validação de coordenadas geográficas
- ✅ Validação de idades
- ✅ Criação de features temporais (ano, mês, dia da semana, período)
- ✅ Criação de features derivadas (faixa etária, crime violento, fim de semana)
- ✅ Codificação de variáveis categóricas
- ✅ Exportação do dataset processado